# 🏗️ E-Commerce Data Lakehouse — Demo Notebook

**Mục tiêu notebook này:**
1. **EDA (Exploratory Data Analysis)** — khám phá dữ liệu qua từng layer Bronze → Silver → Gold
2. **Time Travel** — đọc lại dữ liệu ở version cũ, so sánh sự thay đổi theo thời gian
3. **DESCRIBE HISTORY** — xem toàn bộ lịch sử thay đổi của Delta table

---
> ⚠️ **Yêu cầu:** Đã chạy `python generate_data.py` và `python pipeline.py` trước khi mở notebook này.

## 0. Setup — Khởi tạo SparkSession

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from config.spark_session import get_spark

spark = get_spark("Demo_Notebook")
print(f"Spark version : {spark.version}")
print(f"Delta Lake    : OK")

# Đường dẫn các layer
BRONZE_ORDERS    = "../data/lakehouse/bronze/orders"
SILVER_ORDERS    = "../data/lakehouse/silver/orders"
SILVER_CUSTOMERS = "../data/lakehouse/silver/customers"
GOLD_FACT        = "../data/lakehouse/gold/fact_orders"
GOLD_AGG         = "../data/lakehouse/gold/agg_revenue_daily"

plt.rcParams.update({
    "figure.facecolor" : "white",
    "axes.facecolor"   : "#f8f8f8",
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "font.size"        : 11,
})

---
## 1. EDA — Bronze Layer

Bronze lưu dữ liệu **nguyên trạng** từ nguồn, bao gồm cả bản ghi lỗi và duplicate.  
Mục tiêu EDA tại đây: **đo lường mức độ "bẩn"** của dữ liệu đầu vào.

In [ ]:
df_bronze = spark.read.format("delta").load(BRONZE_ORDERS)

print("=" * 55)
print("BRONZE — Thông tin cơ bản")
print("=" * 55)
print(f"Số bản ghi (bao gồm duplicate) : {df_bronze.count():,}")
print(f"Số cột                          : {len(df_bronze.columns)}")
print()
df_bronze.printSchema()

In [ ]:
print("Mẫu 5 bản ghi từ Bronze:")
df_bronze.select(
    "order_id", "customer_id", "product_id",
    "quantity", "unit_price", "status", "order_date", "ingested_at"
).show(5, truncate=False)

In [ ]:
# -------------------------------------------------------
# Phát hiện Duplicate — đây là vấn đề chính của Data Lake
# -------------------------------------------------------
total_rows   = df_bronze.count()
unique_orders = df_bronze.select("order_id").distinct().count()
dup_count    = total_rows - unique_orders
dup_rate_pct = dup_count / total_rows * 100

print("📊 Phân tích Duplicate tại Bronze layer")
print("-" * 45)
print(f"Tổng bản ghi     : {total_rows:,}")
print(f"Unique order_id  : {unique_orders:,}")
print(f"Số duplicate     : {dup_count:,}")
print(f"Tỷ lệ duplicate  : {dup_rate_pct:.1f}%")
print()
print("⚠️  Đây là lý do Bronze KHÔNG thể dùng trực tiếp cho Analytics!")

# Top 5 order bị duplicate nhiều nhất
print("\nTop 5 order_id bị duplicate nhiều nhất:")
(
    df_bronze.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
    .show(5)
)

In [ ]:
# -------------------------------------------------------
# Null analysis — kiểm tra từng cột
# -------------------------------------------------------
key_cols = ["order_id", "customer_id", "product_id", "quantity", "unit_price", "status", "order_date"]

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in key_cols
])

print("📊 Số lượng NULL theo cột (Bronze):")
null_counts.show()

# Visualize null heatmap
null_pd = null_counts.toPandas().T.reset_index()
null_pd.columns = ["column", "null_count"]

fig, ax = plt.subplots(figsize=(9, 3.5))
colors = ["#e74c3c" if v > 0 else "#2ecc71" for v in null_pd["null_count"]]
bars = ax.barh(null_pd["column"], null_pd["null_count"], color=colors, height=0.55)
ax.set_xlabel("Số bản ghi NULL")
ax.set_title("Null Analysis — Bronze Layer", fontweight="bold", pad=12)
for bar, val in zip(bars, null_pd["null_count"]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=10)
plt.tight_layout()
plt.show()

---
## 2. EDA — Silver Layer

Silver đã qua xử lý: dedup bằng Window function, validate, MERGE INTO.  
Mục tiêu: **xác nhận dữ liệu sạch** và so sánh với Bronze.

In [ ]:
df_silver = spark.read.format("delta").load(SILVER_ORDERS)
df_cust   = spark.read.format("delta").load(SILVER_CUSTOMERS)

n_bronze = df_bronze.count()
n_silver = df_silver.count()

print("=" * 55)
print("SILVER vs BRONZE — So sánh sau khi làm sạch")
print("=" * 55)
print(f"Bronze rows  : {n_bronze:,}  (raw, có duplicate)")
print(f"Silver rows  : {n_silver:,}  (sạch, unique order_id)")
print(f"Đã loại bỏ   : {n_bronze - n_silver:,} bản ghi ({(n_bronze - n_silver)/n_bronze*100:.1f}%)")
print()
print("Silver schema (typed, strict hơn Bronze):")
df_silver.printSchema()

In [ ]:
# -------------------------------------------------------
# Kiểm tra Silver KHÔNG còn duplicate
# -------------------------------------------------------
dup_check = df_silver.groupBy("order_id").count().filter(F.col("count") > 1).count()
print(f"✅ Số order_id bị duplicate trong Silver: {dup_check}  ({'PASS — Không có duplicate' if dup_check == 0 else 'FAIL!'})")

# Kiểm tra kiểu dữ liệu đã được ép đúng
print("\n✅ Kiểm tra kiểu dữ liệu sau khi ép:")
for field in df_silver.schema.fields:
    print(f"   {field.name:<25} → {field.dataType}")

In [ ]:
# -------------------------------------------------------
# Thống kê phân phối đơn hàng theo status
# -------------------------------------------------------
status_df = (
    df_silver.groupBy("status")
    .agg(
        F.count("*").alias("so_don"),
        F.round(F.sum("total_amount") / 1e6, 2).alias("doanh_thu_trieu"),
    )
    .orderBy("status")
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ số đơn theo status
colors_status = ["#3498db", "#2ecc71", "#e74c3c", "#f39c12", "#9b59b6"]
axes[0].bar(status_df["status"], status_df["so_don"],
            color=colors_status[:len(status_df)], width=0.6)
axes[0].set_title("Số đơn hàng theo Status", fontweight="bold")
axes[0].set_xlabel("Status")
axes[0].set_ylabel("Số đơn")
for i, (_, row) in enumerate(status_df.iterrows()):
    axes[0].text(i, row["so_don"] + 2, str(row["so_don"]), ha="center", fontsize=10)

# Biểu đồ doanh thu theo status
axes[1].bar(status_df["status"], status_df["doanh_thu_trieu"],
            color=colors_status[:len(status_df)], width=0.6)
axes[1].set_title("Doanh thu theo Status (triệu VNĐ)", fontweight="bold")
axes[1].set_xlabel("Status")
axes[1].set_ylabel("Triệu VNĐ")

plt.suptitle("Silver Layer — Phân phối đơn hàng", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(status_df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# Xu hướng đơn hàng theo tháng
# -------------------------------------------------------
monthly_df = (
    df_silver
    .withColumn("month", F.date_format("order_date", "yyyy-MM"))
    .groupBy("month")
    .agg(
        F.count("*").alias("so_don"),
        F.round(F.sum("total_amount") / 1e6, 1).alias("doanh_thu_trieu"),
    )
    .orderBy("month")
    .toPandas()
)

fig, ax1 = plt.subplots(figsize=(13, 4.5))
ax2 = ax1.twinx()

x = range(len(monthly_df))
ax1.bar(x, monthly_df["so_don"], color="#3498db", alpha=0.7, width=0.6, label="Số đơn")
ax2.plot(x, monthly_df["doanh_thu_trieu"], color="#e74c3c",
         linewidth=2.2, marker="o", markersize=5, label="Doanh thu (triệu)")

ax1.set_xticks(list(x))
ax1.set_xticklabels(monthly_df["month"], rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Số đơn hàng", color="#3498db")
ax2.set_ylabel("Doanh thu (triệu VNĐ)", color="#e74c3c")
ax1.set_title("Xu hướng đơn hàng & Doanh thu theo tháng — Silver Layer",
              fontweight="bold", pad=12)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

---
## 3. EDA — Gold Layer

Gold là dữ liệu **sẵn sàng cho Analytics** — đã join đầy đủ, có business rules.  
Đây là layer mà Analytics team thực sự query.

In [ ]:
df_fact = spark.read.format("delta").load(GOLD_FACT)
df_agg  = spark.read.format("delta").load(GOLD_AGG)

print("=" * 55)
print("GOLD — fact_orders")
print("=" * 55)
print(f"Số bản ghi : {df_fact.count():,}")
print(f"Partitions : order_year × order_month")
print()
df_fact.select(
    "order_id", "order_date", "customer_name", "region",
    "product_name", "category", "total_amount", "is_high_value", "status"
).show(5, truncate=28)

In [ ]:
# -------------------------------------------------------
# Doanh thu & số đơn theo Region
# -------------------------------------------------------
region_df = (
    df_agg.groupBy("region")
    .agg(
        F.sum("so_don_hang").alias("so_don"),
        F.round(F.sum("doanh_thu") / 1e6, 1).alias("doanh_thu_trieu"),
        F.round(F.avg("gia_tri_trung_binh") / 1e3, 1).alias("avg_nghin_vnd"),
    )
    .orderBy(F.col("doanh_thu_trieu").desc())
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

region_colors = ["#e74c3c", "#3498db", "#2ecc71"]
axes[0].bar(region_df["region"], region_df["doanh_thu_trieu"],
            color=region_colors[:len(region_df)], width=0.5)
axes[0].set_title("Doanh thu theo Region (triệu VNĐ)", fontweight="bold")
for i, v in enumerate(region_df["doanh_thu_trieu"]):
    axes[0].text(i, v + 0.5, f"{v:.1f}M", ha="center", fontsize=10)

axes[1].pie(
    region_df["so_don"],
    labels=region_df["region"],
    autopct="%1.1f%%",
    colors=region_colors[:len(region_df)],
    startangle=90,
    wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[1].set_title("Phân bổ đơn hàng theo Region", fontweight="bold")

plt.suptitle("Gold Layer — Phân tích theo Region", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(region_df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# Top category theo doanh thu
# -------------------------------------------------------
cat_df = (
    df_agg.groupBy("category")
    .agg(
        F.sum("so_don_hang").alias("so_don"),
        F.round(F.sum("doanh_thu") / 1e6, 1).alias("doanh_thu_trieu"),
        F.sum("tong_san_pham_ban").alias("tong_sp"),
    )
    .orderBy(F.col("doanh_thu_trieu").desc())
    .toPandas()
)

cat_colors = ["#9b59b6", "#3498db", "#2ecc71", "#f39c12", "#e74c3c"]
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(cat_df["category"][::-1], cat_df["doanh_thu_trieu"][::-1],
               color=cat_colors[:len(cat_df)], height=0.55)
ax.set_xlabel("Doanh thu (triệu VNĐ)")
ax.set_title("Doanh thu theo Category sản phẩm", fontweight="bold", pad=12)
for bar, val in zip(bars, cat_df["doanh_thu_trieu"][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f"{val:.1f}M", va="center", fontsize=10)
plt.tight_layout()
plt.show()

print(cat_df.to_string(index=False))

In [ ]:
# -------------------------------------------------------
# Business Rule: High Value Orders (> 1 triệu VNĐ)
# -------------------------------------------------------
hv_df = (
    df_fact.groupBy("is_high_value")
    .agg(
        F.count("*").alias("so_don"),
        F.round(F.sum("total_amount") / 1e6, 1).alias("doanh_thu_trieu"),
        F.round(F.avg("total_amount") / 1e3, 1).alias("avg_nghin_vnd"),
    )
    .toPandas()
)

hv_df["label"] = hv_df["is_high_value"].map({True: "High Value (≥1M)", False: "Normal (<1M)"})
print("📊 Phân tích High Value Orders:")
print(hv_df[["label", "so_don", "doanh_thu_trieu", "avg_nghin_vnd"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(hv_df["label"], hv_df["doanh_thu_trieu"],
       color=["#e74c3c", "#bdc3c7"], width=0.45)
ax.set_title("Doanh thu: High Value vs Normal Orders", fontweight="bold")
ax.set_ylabel("Doanh thu (triệu VNĐ)")
for i, v in enumerate(hv_df["doanh_thu_trieu"]):
    ax.text(i, v + 0.5, f"{v:.1f}M", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. DESCRIBE HISTORY — Lịch sử version Delta Table

Đây là tính năng đặc trưng của **Delta Lake** — mọi thay đổi đều được ghi lại trong transaction log.  
Lệnh `DESCRIBE HISTORY` (hoặc `.history()`) cho phép audit toàn bộ lịch sử của table.

In [ ]:
# -------------------------------------------------------
# DESCRIBE HISTORY — Silver orders
# -------------------------------------------------------
silver_table = DeltaTable.forPath(spark, SILVER_ORDERS)

print("📜 DESCRIBE HISTORY — Silver orders")
print("=" * 65)
(
    silver_table.history()
    .select("version", "timestamp", "operation", "operationMetrics", "userName")
    .show(10, truncate=60)
)

# Dùng Spark SQL (cách khác, gần với SQL hơn)
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_ORDERS}`") \
     .select("version", "timestamp", "operation") \
     .show(10, truncate=False)

In [ ]:
# -------------------------------------------------------
# DESCRIBE HISTORY — Gold fact_orders
# -------------------------------------------------------
gold_table = DeltaTable.forPath(spark, GOLD_FACT)

print("📜 DESCRIBE HISTORY — Gold fact_orders")
print("=" * 65)
gold_table.history() \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .show(10, truncate=80)

# Thống kê tổng số version hiện có
n_versions = gold_table.history().count()
print(f"\n📌 Tổng số version trong Gold fact_orders: {n_versions}")

---
## 5. Time Travel — Đọc dữ liệu theo version cũ

**Time Travel** là tính năng cho phép đọc lại dữ liệu tại **bất kỳ version hoặc timestamp nào** trong quá khứ.  
Trong thực tế, tính năng này dùng để:
- **Audit:** "Dữ liệu trông như thế nào trước khi chạy pipeline lúc 3 giờ sáng?"
- **Rollback:** Phục hồi về trạng thái trước khi có lỗi
- **Reproducibility:** Đảm bảo ML model training luôn dùng cùng snapshot data

In [ ]:
# -------------------------------------------------------
# Time Travel theo VERSION NUMBER
# -------------------------------------------------------

# Lấy danh sách version hiện có
history_pd = silver_table.history().select("version", "timestamp", "operation").toPandas()
print("📋 Các version hiện có của Silver orders:")
print(history_pd.to_string(index=False))

latest_version = int(history_pd["version"].max())
print(f"\n🔖 Version hiện tại (latest): {latest_version}")

In [ ]:
# -------------------------------------------------------
# Đọc dữ liệu tại version 0 (snapshot đầu tiên)
# -------------------------------------------------------
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(SILVER_ORDERS)
)

df_latest = spark.read.format("delta").load(SILVER_ORDERS)

n_v0     = df_v0.count()
n_latest = df_latest.count()

print("⏪ Time Travel: So sánh version 0 vs version hiện tại")
print("-" * 50)
print(f"Version 0 (snapshot đầu tiên) : {n_v0:,} rows")
print(f"Version {latest_version} (hiện tại)          : {n_latest:,} rows")
print(f"Chênh lệch                    : {n_latest - n_v0:,} rows")

print("\nMẫu 3 bản ghi tại version 0:")
df_v0.select("order_id", "status", "total_amount", "order_date").show(3)

In [ ]:
# -------------------------------------------------------
# Đọc dữ liệu theo TIMESTAMP
# (truy vấn dữ liệu tại một thời điểm cụ thể trong quá khứ)
# -------------------------------------------------------
from datetime import datetime, timedelta

# Lấy timestamp của version 0 từ history
ts_v0 = history_pd[history_pd["version"] == 0]["timestamp"].values[0]
ts_v0_str = str(ts_v0)[:19]   # format: "yyyy-MM-dd HH:mm:ss"

print(f"🕐 Timestamp của version 0: {ts_v0_str}")

try:
    df_by_ts = (
        spark.read
        .format("delta")
        .option("timestampAsOf", ts_v0_str)
        .load(SILVER_ORDERS)
    )
    print(f"✅ Đọc theo timestamp thành công: {df_by_ts.count():,} rows")
    print("→ Kết quả trùng khớp với versionAsOf=0:",
          df_by_ts.count() == df_v0.count())
except Exception as e:
    print(f"⚠️  Timestamp query: {e}")
    print("   (Bình thường nếu chỉ có 1 version — timestamp = version 0)")

In [ ]:
# -------------------------------------------------------
# Phân tích sự khác biệt giữa 2 version (nếu có >= 2)
# -------------------------------------------------------
if latest_version >= 1:
    df_v1 = (
        spark.read
        .format("delta")
        .option("versionAsOf", 1)
        .load(SILVER_ORDERS)
    )

    # Tìm các order_id có mặt trong v1 nhưng không có trong v0 (bản ghi mới)
    new_in_v1 = df_v1.join(df_v0, on="order_id", how="left_anti")
    print(f"\n🆕 Bản ghi mới xuất hiện ở version 1 (chưa có ở version 0): {new_in_v1.count():,}")

    # Tìm các order_id có status thay đổi giữa 2 version
    changed = (
        df_v0.select("order_id", F.col("status").alias("status_v0"))
        .join(df_v1.select("order_id", F.col("status").alias("status_v1")), on="order_id")
        .filter(F.col("status_v0") != F.col("status_v1"))
    )
    print(f"🔄 Đơn hàng có status thay đổi giữa v0 → v1: {changed.count():,}")
    if changed.count() > 0:
        changed.show(5)
else:
    print("ℹ️  Chỉ có 1 version. Chạy lại pipeline để tạo thêm version và demo diff.")

---
## 6. Demo nâng cao: Rollback & Re-run Idempotency

Đây là 2 tính năng thực chiến quan trọng mà ACID của Delta Lake mang lại.

In [ ]:
# -------------------------------------------------------
# Demo: RESTORE về version cũ (Rollback)
# -------------------------------------------------------
# Trong production: dùng khi phát hiện pipeline ghi dữ liệu sai
#
# ⚠️ CẢNH BÁO: Cell này sẽ thực sự restore table về version 0
#    Hãy chỉ chạy nếu muốn demo, sau đó chạy lại pipeline.py để restore.
# 
# Bỏ comment để chạy:

# silver_table.restoreToVersion(0)
# print("✅ Đã rollback Silver orders về version 0")
# print(f"   Số bản ghi hiện tại: {spark.read.format('delta').load(SILVER_ORDERS).count():,}")

print("⏸️  Rollback demo bị tắt để tránh ảnh hưởng dữ liệu.")
print("   Bỏ comment 2 dòng trên để chạy thực tế.")
print("   Lệnh: DeltaTable.forPath(spark, path).restoreToVersion(0)")

In [ ]:
# -------------------------------------------------------
# Demo: Idempotency — chạy lại pipeline không tạo duplicate
# -------------------------------------------------------
print("🔁 Demo Idempotency — MERGE INTO đảm bảo chạy lại N lần vẫn cho cùng kết quả\n")

before = spark.read.format("delta").load(SILVER_ORDERS).count()
print(f"Trước khi chạy lại Silver: {before:,} rows")

# Chạy lại silver_orders với CÙNG Bronze data
from silver.silver_orders import run as silver_orders_run
silver_orders_run()

after = spark.read.format("delta").load(SILVER_ORDERS).count()
print(f"\nSau khi chạy lại Silver : {after:,} rows")
print(f"Chênh lệch              : {after - before:,} rows")
print(f"\n{'✅ PASS' if after == before else '❌ FAIL'} — ",
      "Số bản ghi không thay đổi → MERGE INTO idempotent" if after == before
      else "Có bản ghi mới → pipeline KHÔNG idempotent")

---
## 7. Tổng kết — So sánh 3 Layer

In [ ]:
# -------------------------------------------------------
# Bảng tổng kết toàn bộ pipeline
# -------------------------------------------------------
n_bronze  = df_bronze.count()
n_silver  = spark.read.format("delta").load(SILVER_ORDERS).count()
n_fact    = spark.read.format("delta").load(GOLD_FACT).count()
n_agg     = spark.read.format("delta").load(GOLD_AGG).count()
n_cust    = spark.read.format("delta").load(SILVER_CUSTOMERS).count()

total_revenue = (
    spark.read.format("delta").load(GOLD_AGG)
    .agg(F.sum("doanh_thu")).collect()[0][0]
)

print("=" * 60)
print("  TỔNG KẾT PIPELINE — E-Commerce Data Lakehouse")
print("=" * 60)
print(f"")
print(f"  🔶 BRONZE")
print(f"     orders (raw + dup)  : {n_bronze:>8,} rows")
print(f"")
print(f"  ⬜ SILVER")
print(f"     orders (clean)      : {n_silver:>8,} rows  ← {n_bronze - n_silver:,} dup removed")
print(f"     customers (clean)   : {n_cust:>8,} rows")
print(f"")
print(f"  🟡 GOLD")
print(f"     fact_orders         : {n_fact:>8,} rows")
print(f"     agg_revenue_daily   : {n_agg:>8,} rows  ← (ngày × region × category)")
print(f"     Tổng doanh thu      : {total_revenue/1e9:>8.2f} tỷ VNĐ")
print(f"")
print(f"  📜 DELTA LAKE FEATURES USED")
print(f"     ACID / MERGE INTO   : ✅  Silver orders upsert")
print(f"     Schema enforcement  : ✅  Silver layer strict schema")
print(f"     Time Travel         : ✅  versionAsOf / timestampAsOf")
print(f"     DESCRIBE HISTORY    : ✅  audit log mọi thay đổi")
print(f"     Partition pruning   : ✅  Gold partitioned by year/month")
print(f"     Structured Streaming: ✅  Bronze events ingest")
print("=" * 60)

In [ ]:
# -------------------------------------------------------
# Biểu đồ tóm tắt: Data flow qua các layer
# -------------------------------------------------------
layers  = ["Bronze\n(raw)", "Silver\n(clean)", "Gold Fact\n(enriched)", "Gold Agg\n(analytics)"]
counts  = [n_bronze, n_silver, n_fact, n_agg]
colors  = ["#f39c12", "#bdc3c7", "#f1c40f", "#2ecc71"]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(layers, counts, color=colors, width=0.5, edgecolor="white", linewidth=1.5)
ax.set_ylabel("Số bản ghi")
ax.set_title("Data Flow: Số bản ghi qua từng Layer", fontweight="bold", pad=14)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:,}", ha="center", fontsize=11, fontweight="bold")

# Thêm annotation mũi tên
for i in range(len(layers) - 1):
    ax.annotate("", xy=(i + 0.75, max(counts) * 0.5),
                xytext=(i + 0.25, max(counts) * 0.5),
                arrowprops=dict(arrowstyle="->", color="#7f8c8d", lw=1.5))

plt.tight_layout()
plt.show()

print("\n✅ Demo hoàn tất! Xem các cell trên để hiểu chi tiết từng bước.")